In [1]:
import getpass
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage
from IPython.display import Image, display
from langgraph.graph import MessagesState
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode, tools_condition
import os

load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")

from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2
)

d:\Learning\AI-Agents\tf_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
from langchain_tavily import TavilySearch
from langchain_community.document_loaders import WebBaseLoader

tavily_search = TavilySearch(
    max_results = 3,
    search_depth = "advanced",
    include_domains= ["who.int"]
)

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [3]:
question = "What is the latest WHO report on malaria?"
search_results = tavily_search.invoke(question)

In [4]:
search_results

{'query': 'What is the latest WHO report on malaria?',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://www.who.int/teams/global-malaria-programme/reports/world-malaria-report-2025',
   'title': 'World malaria report 2025 - World Health Organization (WHO)',
   'content': 'Since 2000, 2.3 billion malaria cases and 14 million deaths have been averted worldwide – including 1 million lives saved in 2024 alone – and',
   'score': 0.77137923,
   'raw_content': None},
  {'url': 'https://www.afro.who.int/health-topics/malaria',
   'title': 'Malaria | WHO | Regional Office for Africa',
   'content': 'decreased from 808 000 in 2000 to 548 000 in 2017, before increasing to 604 000 in 2020. Estimated deaths decreased again to 580 000 in 2022. The malaria mortality rate decreased by 60% between 2000 and 2019, from 143 to 57 deaths per 100 000 population at risk, before rising to 61 in 2020 and decreasing again to 56 in 2022.  Cabo Verde has reported zero m

In [6]:
scraped_docs = []
urls = [result.get("url") for result in search_results["results"] if result.get("url")]
print(f"Found urls {urls}")

if not urls:
    print("No valid URLs in search results.")

for url in urls:
    try:
        print(f"Scrapping {url}...")
        loader = WebBaseLoader(url)
        docs = loader.load()
        for doc in docs:
            doc.metadata["source"] = url
        scraped_docs.extend(docs)
    except Exception as e:
        print(f"Error Scrapping {url}: {e}")

Found urls ['https://www.who.int/teams/global-malaria-programme/reports/world-malaria-report-2025', 'https://www.afro.who.int/health-topics/malaria', 'https://www.who.int/campaigns/world-malaria-day/2026']
Scrapping https://www.who.int/teams/global-malaria-programme/reports/world-malaria-report-2025...
Scrapping https://www.afro.who.int/health-topics/malaria...
Scrapping https://www.who.int/campaigns/world-malaria-day/2026...


In [7]:
scraped_docs[1]

Document(metadata={'source': 'https://www.afro.who.int/health-topics/malaria', 'title': 'Malaria | WHO | Regional Office for Africa', 'description': 'The World Health Organization (WHO) is building a better future for people everywhere. Health lays the foundation for vibrant and productive communities, stronger economies, safer nations and a better world. Our work touches lives around the world every day – often in invisible ways. As the lead health authority within the United Nations (UN) system, we help ensure the safety of the air we breathe, the food we eat, the water we drink and the medicines and vaccines that treat and protect us. The Organization aims to provide every child, woman and man with the best chance to lead a healthier, longer life.', 'language': 'en'}, page_content="\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nMalaria | WHO | Regional Office for Africa\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n      Skip to 

In [8]:
from langchain_core.documents import Document
from typing import List
from typing_extensions import TypedDict

In [ ]:
class GraphState(TypedDict):
    question: str
    documents: List[Document]
    sender: str

In [ ]:
def web_search_node(state):
    """Searches the web for information, then scrapes the content from the resulting URLs."""
    print("---- Calling Web Search Node ---")
    question = state["question"]

    tavily_search = TavilySearch(
        max_results = 3, 
        search_depth = "advanced",
        include_domains= ["who.int"]
    )

    search_results = tavily_search.invoke(question)

    scraped_docs = []
    if not search_results or "results" not in search_results:
        print("No search results found.")
        return {"documents": [], "sender": "web_search_node"}

    urls = [result.get("url") for result in search_results["results"] if result.get("url")]
    print(f"Found urls {urls}")

    if not urls:
        print("No valid URLs in search results.")
        return {"documents": [], "sender": "web_search_node"}

    for url in urls:
        try:
            print(f"Scrapping {url}...")
            loader = WebBaseLoader(url)
            docs = loader.load()
            for doc in docs:
                doc.metadata["source"] = url
            scraped_docs.extend(docs)
        except Exception as e:
            print(f"Error Scrapping {url}: {e}")